# Clustering Based Data Analytics using K-Means and SOM

**Objective:** Apply clustering techniques to discover natural groups in numeric data using K-Means and a Self-Organizing Map workflow.

**Dataset:** Iris dataset or customer segmentation dataset

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn minisom


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42


In [ ]:
iris = load_iris(as_frame=True)
data = iris.frame.copy()
X = data[iris.feature_names]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_scaled)
    rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X_scaled, labels),
        }
    )

metrics = pd.DataFrame(rows)
display(metrics)
metrics.to_csv(RESULTS_DIR / "cluster_metrics.csv", index=False)


In [ ]:
best_k = int(metrics.sort_values("silhouette", ascending=False).iloc[0]["k"])
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
labels = kmeans.fit_predict(X_scaled)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)

cluster_output = pd.DataFrame(coords, columns=["pc1", "pc2"])
cluster_output["cluster"] = labels
cluster_output["target"] = iris.target
cluster_output.to_csv(RESULTS_DIR / "cluster_assignments.csv", index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=cluster_output, x="pc1", y="pc2", hue="cluster", palette="viridis", ax=axes[0])
axes[0].set_title("K-Means Clusters in PCA Space")
axes[0].grid(alpha=0.25)

axes[1].plot(metrics["k"], metrics["inertia"], marker="o", label="Inertia")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Inertia")
axes[1].set_title("Elbow Analysis")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "clustering_dashboard.png", dpi=180)
plt.show()
